In [1]:
import numpy as np
import torch as th
import os
import os.path as osp
import pickle
import warnings
warnings.filterwarnings("ignore")
from scripts_utils import Parser
import diffuser.utils as utils
from diffuser.utils.arrays import to_np
from diffuser.datasets import object_rearrangement
from diffuser.datasets import AGENT
from AGENT_env import AGENT_env
from diffuser.datasets import highway
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from highway_env import register_highway_envs

register_highway_envs()

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
pybullet build time: Nov 28 2023 23:51:11


In [2]:
%%bash
env="Overcooked"

# asymmetric_advantages coordination_ring counter_circuit_o_1order=random3, cramped_room forced_coordination=random0
layout="counter_circuit_o_1order" # asymmetric_advantages diverse_counter_circuit_6x5
pop=${layout}_comedi

num_agents=2
algo="population"
agent0_policy_name="comedi_oracle"
agent1_policy_name="proxy"
exp="eval-${agent0_policy_name}-${agent1_policy_name}"

path=/mmfs1/gscratch/cse/jiayiy9/GAMMA-human-ai-collaboration/mapbt/scripts/overcooked_population
population_yaml_path=${path}/pop_data/${pop}/comedi_oracle_vs_proxy.yml

export POLICY_POOL=${path}

In [3]:
from mapbt.algorithms.population.policy_pool import PolicyPool as Policy


In [4]:
def eval_overcooked(
    basedir, diffusion, dataset, renderer, dummy_cond, all_cond_features, all_cond_text,
    condition_guidance_w, device
):
    """
    Evaluate the overcooked model.
    """

In [5]:
if __name__ == "__main__":
    # args = Parser().parse_args('plan')
    device = th.device('cpu' if not th.cuda.is_available() else 'cuda')

    # load bc proxy
    population_yaml_path = "/mmfs1/gscratch/cse/jiayiy9/GAMMA-human-ai-collaboration/mapbt/scripts/overcooked_population/pop_data/counter_circuit_o_1order_comedi/comedi_oracle_vs_proxy.yml"
    policy = Policy(None, None, None, None, device=device)
    featurize_type = policy.load_population(population_yaml_path, evaluation=True)
    # policy.policy_pool['proxy'] is EvalPolicy object
    proxy_policy = policy.policy_pool['proxy']
    print("featurize_type: ", featurize_type)

featurize_type:  {'comedi_oracle': 'ppo', 'proxy': 'bc'}


In [6]:
def parse_args(args, parser):
    parser.add_argument("--old_dynamics", default=False, action='store_true', help="old_dynamics in mdp")
    parser.add_argument("--layout_name", type=str, default='cramped_room', help="Name of Submap, 40+ in choice. See /src/data/layouts/.")
    parser.add_argument('--num_agents', type=int,
                        default=1, help="number of players")
    parser.add_argument("--initial_reward_shaping_factor", type=float, default=1.0, help="Shaping factor of potential dense reward.")
    parser.add_argument("--reward_shaping_factor", type=float, default=1.0, help="Shaping factor of potential dense reward.")
    parser.add_argument("--reward_shaping_horizon", type=int, default=2.5e6, help="Shaping factor of potential dense reward.")
    parser.add_argument("--use_phi", default=False, action='store_true', help="While existing other agent like planning or human model, use an index to fix the main RL-policy agent.")  
    parser.add_argument("--use_hsp", default=False, action='store_true')   
    parser.add_argument("--random_index", default=False, action='store_true')
    parser.add_argument("--use_agent_policy_id", default=False, action='store_true', help="Add policy id into share obs, default False")
    parser.add_argument("--overcooked_version", default="old", type=str, choices=["new", "old"])
    parser.add_argument("--use_detailed_rew_shaping", default=False, action='store_true')
    parser.add_argument("--random_start_prob", default=0., type=float)
    parser.add_argument("--store_traj", default=False, action='store_true')
    # population
    parser.add_argument("--population_yaml_path", type=str, help="Path to yaml file that stores the population info.")
    
    # overcooked evaluation
    parser.add_argument("--agent0_policy_name", type=str, help="policy name of agent 0")
    parser.add_argument("--agent1_policy_name", type=str, help="policy name of agent 1")

    all_args = parser.parse_known_args(args)[0]

    return all_args

In [7]:
from mapbt.config import get_config
from mapbt.envs.overcooked.Overcooked_Env import Overcooked
import sys
parser = get_config()
args = sys.argv[1:]
all_args = parse_args(args, parser)

# assert all_args.algorithm_name == "population"
run_dir = '/mmfs1/gscratch/cse/jiayiy9/GAMMA-human-ai-collaboration/mapbt/scripts/results/Overcooked/counter_circuit_o_1order/population/eval-comedi_oracle-proxy/run14'

/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/site-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/mmfs1/gscratch/cse/jiayiy9/miniconda3/envs/ftl-igm/lib/python3.8/site-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)


In [8]:
all_args.layout_name = 'counter_circuit_o_1order'
all_args.num_agents = 2
from argparse import Namespace
all_args = Namespace(activation_id=1, agent0_policy_name='comedi_oracle', agent1_policy_name='proxy', algorithm_name='population', attn_N=1, attn_heads=4, attn_size=64, aux_epoch=5, clip_param=0.2, clone_coef=1.0, cnn_layers_params=None, critic_lr=0.0005, critic_warmup_horizon=0, cuda=True, cuda_deterministic=True, data_chunk_length=10, dropout=0.0, entropy_coef=0.01, env_name='Overcooked', episode_length=400, eval_episodes=3, eval_interval=25, eval_stochastic=True, experiment_name='eval-comedi_oracle-proxy', explored_ratio_threshold=0.9, gae_lambda=0.95, gain=0.01, gamma=0.99, hidden_size=64, huber_delta=10.0, ifi=0.1, influence_layer_N=1, initial_reward_shaping_factor=1.0, layer_N=1, layout_name='counter_circuit_o_1order', log_interval=5, lr=0.0005, max_grad_norm=10.0, mlp_hidden_size=64, model_dir=None, n_eval_rollout_threads=3, n_render_rollout_threads=1, n_rollout_threads=32, n_training_threads=1, num_agents=2, num_env_steps=10000000.0, num_mini_batch=1, num_v_out=1, old_dynamics=True, opti_eps=1e-05, overcooked_version='old', policy_value_loss_coef=1, population_yaml_path='./overcooked_population/pop_data/counter_circuit_o_1order_comedi/comedi_oracle_vs_proxy.yml', ppo_epoch=15, random_index=False, random_start_prob=0.0, recurrent_N=1, render_episodes=5, reward_shaping_factor=1.0, reward_shaping_horizon=2500000.0, save_gifs=False, save_interval=1, seed=1, share_policy=True, stacked_frames=1, store_traj=False, tau=0.995, use_agent_policy_id=False, use_attn=False, use_attn_internal=True, use_average_pool=True, use_cat_self=True, use_centralized_V=True, use_clipped_value_loss=True, use_conv1d=False, use_detailed_rew_shaping=False, use_eval=False, use_feature_normalization=True, use_gae=True, use_hsp=False, use_huber_loss=True, use_influence_policy=False, use_linear_lr_decay=False, use_max_grad_norm=True, use_maxpool2d=False, use_naive_recurrent_policy=False, use_obs_instead_of_state=False, use_orthogonal=True, use_phi=False, use_policy_active_masks=True, use_policy_vhead=False, use_popart=False, use_proper_time_limits=False, use_recurrent_policy=True, use_render=False, use_single_network=False, use_stacked_frames=False, use_value_active_masks=True, use_valuenorm=True, use_wandb=False, user_name='user_name', value_loss_coef=1, wandb_name='wandb_name', wandb_tags=[], weight_decay=0)

In [33]:

def make_eval_env(all_args, run_dir, nenvs=3):
    def get_env_fn(rank):
        def init_env():
            if all_args.env_name == "Overcooked":
                env = Overcooked(all_args, run_dir, rank=rank)
            else:
                print("Can not support the " +
                      all_args.env_name + "environment.")
                raise NotImplementedError
            env.seed(all_args.seed * 50000 + rank * 10000)
            return env
        return init_env
    return ChooseSubprocVecEnv([get_env_fn(i) for i in range(nenvs)])

envs = make_eval_env(all_args, run_dir)
featurize_type = [['ppo', 'bc'], ['ppo', 'bc'], ['ppo', 'bc']]
envs.reset_featurize_type(featurize_type)

Using OvercookedEnv with the following parameters:
Using OvercookedEnv with the following parameters:Namespace(activation_id=1, agent0_policy_name='comedi_oracle', agent1_policy_name='proxy', algorithm_name='population', attn_N=1, attn_heads=4, attn_size=64, aux_epoch=5, clip_param=0.2, clone_coef=1.0, cnn_layers_params=None, critic_lr=0.0005, critic_warmup_horizon=0, cuda=True, cuda_deterministic=True, data_chunk_length=10, dropout=0.0, entropy_coef=0.01, env_name='Overcooked', episode_length=400, eval_episodes=3, eval_interval=25, eval_stochastic=True, experiment_name='eval-comedi_oracle-proxy', explored_ratio_threshold=0.9, gae_lambda=0.95, gain=0.01, gamma=0.99, hidden_size=64, huber_delta=10.0, ifi=0.1, influence_layer_N=1, initial_reward_shaping_factor=1.0, layer_N=1, layout_name='counter_circuit_o_1order', log_interval=5, lr=0.0005, max_grad_norm=10.0, mlp_hidden_size=64, model_dir=None, n_eval_rollout_threads=3, n_render_rollout_threads=1, n_rollout_threads=32, n_training_threa

In [34]:
obs, shared_obs, obs_avail = envs.reset([ True,  True,  True])

    
obs[0]

(array([[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        ...,
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]],
 
        [[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0,

In [35]:
proxy_policy.reset(num_envs=all_args.n_eval_rollout_threads, num_agents=2)
for e in range(all_args.n_eval_rollout_threads):
    proxy_policy.register_control_agent(e=e, a=1)

In [36]:
obs_lst = [obs[e][a] for (e, a) in proxy_policy.control_agents]

In [37]:
obs_lst

[array([ 1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  3.,  0.,  0., -3.,
         1.,  0.,  0.,  0.,  0.,  4.,  1.,  0.,  0.,  1.,  1.,  0.,  0.,
         0.,  0.,  0.,  0.,  0., -1.,  1.,  1.,  0.,  0.,  0.,  0.,  0.,
         0.,  1., -1.,  1.,  1.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  1.,  0.,  0., -3., -1.,  0.,  0.,  0.,  0.,  4.,
        -1.,  0.,  0.,  1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -3.,
         1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  1., -3.,  1.,  1.,  0.,
         0.,  0.,  2.,  3.,  1.]),
 array([ 1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  3.,  0.,  0., -3.,
         1.,  0.,  0.,  0.,  0.,  4.,  1.,  0.,  0.,  1.,  1.,  0.,  0.,
         0.,  0.,  0.,  0.,  0., -1.,  1.,  1.,  0.,  0.,  0.,  0.,  0.,
         0.,  1., -1.,  1.,  1.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  1.,  0.,  0., -3., -1.,  0.,  0.,  0.,  0.,  4.,
        -1.,  0.,  0.,  1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -3.,
         1.,  1.

In [42]:
agents = proxy_policy.control_agents
proxy_policy.to('cuda:0')
proxy_action = proxy_policy.step(np.stack(obs_lst, axis=0), agents)
print(proxy_action)

[[4]
 [2]
 [4]]


In [45]:
diffusion_loadpath='/mmfs1/gscratch/cse/jiayiy9/ftl-igm-overcooked/code/logs/overcooked/diffusion/defaults_H32_T100/20250415-114236'
diffusion_experiment = utils.load_diffusion(
    diffusion_loadpath,
    epoch='latest', seed=None,
)

[ utils/serialization ] Loaded config from /mmfs1/gscratch/cse/jiayiy9/ftl-igm-overcooked/code/logs/overcooked/diffusion/defaults_H32_T100/20250415-114236/dataset_config.pkl

[ utils/config ] Config: <class 'diffuser.datasets.overcooked.OvercookedSequenceDataset'>
    args: {'action_weight': 10,
 'attention': False,
 'batch_size': 32,
 'bucket': None,
 'chunk_length': 64,
 'clip_denoised': True,
 'condition_guidance_w': 1.0,
 'config': 'config.overcooked',
 'dataset': 'overcooked',
 'dataset_path': 'data/overcooked_dataset/counter_circuit_o_1order_mep/dataset.hdf5',
 'device': 'cuda',
 'diffusion': 'models.GaussianDiffusion',
 'dim_mults': (1, 2, 4, 8),
 'domain': 'overcooked',
 'ema_decay': 0.995,
 'episode_length': 400,
 'exp_name': 'diffusion/defaults_H32_T100',
 'gradient_accumulate_every': 2,
 'horizon': 32,
 'learning_rate': 0.0002,
 'loader': 'datasets.OvercookedSequenceDataset',
 'logbase': 'logs',
 'loss_discount': 1,
 'loss_type': 'l2',
 'loss_weights': None,
 'max_path_lengt

Using cache found in /mmfs1/gscratch/cse/jiayiy9/.cache/torch/hub/pytorch_vision_v0.10.0



[ utils/serialization ] Loading model epoch: 0



In [46]:
diffusion = diffusion_experiment.diffusion
diffusion.model.eval()
dataset = diffusion_experiment.dataset
renderer = diffusion_experiment.renderer   

In [48]:
# results path
basedir = diffusion_loadpath

In [49]:

def open_loop_overcooked(basedir, diffusion, dataset, renderer, condition_guidance_w, device, n_demos_eval=10):
    all_samples, all_cond_text, all_inits, all_init_ims, all_gt = [], [], [], [], []
    for _ in range(n_demos_eval):
        sample = dataset.__getitem__(0)
        with th.no_grad():
            # cond: 1 digit id for cooperator
            # dummy_cond: 0
            # cond_obs: past observations, padded with zero
            samples = diffusion.p_sample_loop(
                shape=(1, dataset.horizon, dataset.observation_dim),
                cond=th.unsqueeze(utils.to_torch(sample.conditions).to(device),0),
                dummy_cond=th.unsqueeze(utils.to_torch(sample.dummy_cond).to(device),0),
                cond_obs=th.unsqueeze(utils.to_torch(sample.conditions_obs).to(device),0),
                cond_im=None,
                compose=False,
            )
        all_inits.append(dataset.unnormalize(sample.conditions_obs))
        all_init_ims.append(dataset.unnormalize_im(np.expand_dims(sample.conditions_obs_im,axis=0)).squeeze())
        all_samples.append(dataset.unnormalize(to_np(samples.trajectories)).squeeze())
        all_cond_text.append('') #dummy to plot
        all_gt.append(dataset.unnormalize(to_np(sample.trajectories)).squeeze())
    # save, render
    eval_dir = osp.join(basedir, f'eval_train_w_{condition_guidance_w}')
    # if not osp.isdir(eval_dir): os.makedirs(eval_dir)  
    # with open(osp.join(eval_dir, f'samples.pkl'), 'wb') as f: pickle.dump([all_samples, all_cond_text, all_inits, all_init_ims, all_gt], f)
    # renderer.composite(osp.join(eval_dir, f'render_samples.png'), np.array(all_samples), np.array(all_cond_text), np.array(all_inits), np.array(all_init_ims))
    # renderer.composite(osp.join(eval_dir, f'render_gt.png'), np.array(all_gt), np.array(all_cond_text), np.array(all_inits), np.array(all_init_ims))


In [51]:
device = th.device('cpu' if not th.cuda.is_available() else 'cuda')
open_loop_overcooked(basedir, diffusion, dataset, renderer, 1.0, device)

RuntimeError: Given groups=1, weight of size [128, 1041, 5], expected input[1, 1040, 32] to have 1041 channels, but got 1040 channels instead